# Train Hate Speech Detection on Colab

This notebook clones or updates the project from GitHub, prepares data, then calls the production Python modules in `src/`.

Default branch is `main`. Change `BRANCH` in the first code cell if you want to train from another branch, for example `refactor`.

In [1]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/thong7d/hate-speech-detection.git"
BRANCH = "main"
PROJECT_DIR = Path("/content/hate-speech-detection")

if PROJECT_DIR.exists():
    if not (PROJECT_DIR / ".git").exists():
        raise RuntimeError(f"{PROJECT_DIR} exists but is not a git repository. Remove it or choose another PROJECT_DIR.")
    os.chdir(PROJECT_DIR)
    subprocess.run(["git", "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "checkout", BRANCH], check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
    os.chdir(PROJECT_DIR)

print("Project directory:", Path.cwd())
subprocess.run(["git", "branch", "--show-current"], check=True)

Project directory: /content/hate-speech-detection


CompletedProcess(args=['git', 'branch', '--show-current'], returncode=0)

In [2]:
from pathlib import Path

skip_packages = ("underthesea", "gradio", "pyngrok", "google-generativeai")
source = Path("requirements.txt")
target = Path("requirements-colab.txt")
lines = []
for line in source.read_text(encoding="utf-8").splitlines():
    stripped = line.strip()
    if stripped and not stripped.startswith(skip_packages):
        lines.append(line)
target.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(target.read_text(encoding="utf-8"))

transformers==4.40.2
datasets==2.19.2
torch>=2.1.0
accelerate==0.29.3
scikit-learn==1.4.2
pandas==2.2.2
pyarrow==16.0.0
requests==2.32.2
PyYAML==6.0.2
matplotlib==3.9.0
seaborn==0.13.2
langdetect==1.0.9
fastapi==0.111.0
uvicorn==0.29.0
omegaconf==2.3.0
sentencepiece==0.2.0
protobuf==4.25.3



In [3]:
!python -m pip install -U pip setuptools wheel
# Gỡ bỏ TensorFlow triệt để để ngăn transformers gọi các module gây lỗi protobuf
!python -m pip uninstall -y tensorflow
# Không can thiệp vào protobuf, để Colab dùng bản mặc định
!python -m pip install -r requirements-colab.txt
!python -m pip install huggingface_hub sentencepiece
!python -m pip uninstall -y peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 63.1 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 82.0.1 which is incompatible.
Found existing installation: tensorflow 2.20.0
Uninstalling tensorflow-2.20.0:
  Successfully uninstalled tensorflow-2.20.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 11.5 MB/s  0:00

In [4]:
from pathlib import Path

from src.data.download import download_and_extract
from src.data.preprocessing import process_split

raw_paths = {
    "train": Path("data/raw/vihsd/train.csv"),
    "dev": Path("data/raw/vihsd/dev.csv"),
    "test": Path("data/raw/vihsd/test.csv"),
}
processed_paths = {
    "train": Path("data/processed/train.parquet"),
    "dev": Path("data/processed/dev.parquet"),
    "test": Path("data/processed/test.parquet"),
}

if not all(path.exists() for path in raw_paths.values()):
    download_and_extract("configs/paths.yaml")

for split in ("train", "dev", "test"):
    df = process_split(raw_paths[split], processed_paths[split], use_word_segmentation=False)
    print(f"{split}: {len(df)} rows -> {processed_paths[split]}")

Download successful.
Extracting files...
[OK] Extracted: test.csv -> /content/hate-speech-detection/data/raw/vihsd/test.csv
[OK] Extracted: dev.csv -> /content/hate-speech-detection/data/raw/vihsd/dev.csv
[OK] Extracted: train.csv -> /content/hate-speech-detection/data/raw/vihsd/train.csv
[SUCCESS] Phase 1: Data Acquisition completed successfully.
train: 23833 rows -> data/processed/train.parquet
dev: 2639 rows -> data/processed/dev.parquet
test: 6613 rows -> data/processed/test.parquet


In [5]:
import os

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
except Exception:
    pass

if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"])
else:
    print("Khong du du lieu de xac minh HF_TOKEN hoac HF_TOKEN chua duoc cau hinh")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [6]:
!python -c "import torch, transformers, pandas, pyarrow, sklearn, src; print('OK')"

OK


In [7]:
!python -m src.training.train --config configs/train.yaml

[TRAIN] Final training set size: 56648 samples
[TRAIN] Validation set size: 2639 samples
[TRAIN] Label distribution: {2: 20470, 0: 19970, 1: 16208}
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 117kB/s]
config.json: 100% 615/615 [00:00<00:00, 4.20MB/s]
sentencepiece.bpe.model: 100% 5.07M/5.07M [00:00<00:00, 18.3MB/s]
tokenizer.json: 9.10MB [00:00, 25.1MB/s]
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
model.safetensors: 100% 1.12G/1.12G [00:07<00:00, 154MB/s]
Some weights of XLMRobertaTextCNN were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['convs.0.bias', 'convs.0.weight', 'convs.1.bias', 'convs.1.

In [8]:
!python -m src.evaluation.evaluate --config configs/train.yaml

Exception ignored in: <function _xla_gc_callback at 0x7abde151ee80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/jax/_src/lib/__init__.py", line 127, in _xla_gc_callback
    def _xla_gc_callback(*args):
    
KeyboardInterrupt: 
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/hate-speech-detection/src/evaluation/evaluate.py", line 291, in <module>
    main()
  File "/content/hate-speech-detection/src/evaluation/evaluate.py", line 286, in main
    metrics = evaluate_from_config(config, hf_repo_id=args.hf_repo_id)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

In [9]:
push_flag = "--push-to-hub" if os.environ.get("HF_TOKEN") else ""
!python -m src.export.export_model --config configs/train.yaml {push_flag}

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/hate-speech-detection/src/export/export_model.py", line 142, in <module>
    main()
  File "/content/hate-speech-detection/src/export/export_model.py", line 133, in main
    result = export_from_config(
             ^^^^^^^^^^^^^^^^^^^
  File "/content/hate-speech-detection/src/export/export_model.py", line 70, in export_from_config
    result.update(push_artifact_to_hub(artifact_dir, export_cfg["hf_repo_id"], push_checkpoints=push_checkpoints))
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/hate-speech-detection/src/export/export_model.py", line 97, in push_artifact_to_hub
    upload_folder(
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py", line 114, in _inner_fn
    return fn(*args, **kwargs)
           ^^^^^^^

In [10]:
!python -m src.evaluation.manual_tests --config configs/model.yaml

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/hate-speech-detection/src/evaluation/manual_tests.py", line 8, in <module>
    from src.models.classifier import HateSpeechClassifier
  File "/content/hate-speech-detection/src/models/classifier.py", line 11, in <module>
    from transformers import XLMRobertaPreTrainedModel, XLMRobertaModel, XLMRobertaConfig
  File "/usr/local/lib/python3.12/dist-packages/transformers/__init__.py", line 26, in <module>
    from . import dependency_versions_check
  File "/usr/local/lib/python3.12/dist-packages/transformers/dependency_versions_check.py", line 16, in <module>
object address  : 0x7dc7975287c0
object refcount : 3
object type     : 0xa284e0
object type name: KeyboardInterrupt
object repr     : KeyboardInterrupt()
lost sys.stderr
^C


Outputs:

- Local artifact: `artifacts/hate_speech_model/`
- Final model: `artifacts/hate_speech_model/model/`
- Hugging Face repo: configured in `configs/train.yaml`